# Assignment 06: Clinic Data Wrangling Pipeline (Midterm)

Follow `README.md`: it names every file to save in `output/`, with its first line and line count. Replace each **TODO**, then restart the kernel and **Run All** before you commit, so the saved outputs show your work.

In [1]:
# Supplied setup: run this cell first and leave it unchanged.
from pathlib import Path
import hashlib
import json
import subprocess
import sys

import numpy as np
import pandas as pd

print("Python", sys.version.split()[0], "| NumPy", np.__version__, "| pandas", pd.__version__)


def find_assignment_dir(start):
    """The folder holding data_manifest.json: this one, a parent, or 06/mid-term below one."""
    for folder in (start, *start.parents):
        for candidate in (folder, folder / "06" / "mid-term"):
            if (candidate / "data_manifest.json").is_file():
                return candidate
    raise FileNotFoundError("Open this notebook from the assignment folder that holds data_manifest.json.")


BASE_DIR = find_assignment_dir(Path.cwd().resolve())
DATA_DIR = BASE_DIR / "data"
OUTPUT_DIR = BASE_DIR / "output"
OUTPUT_DIR.mkdir(exist_ok=True)
PROFILE_PATH = OUTPUT_DIR / "table_profile.csv"
AUDIT_PATH = OUTPUT_DIR / "issue_audit.csv"
PATIENTS_SUMMARY_PATH = OUTPUT_DIR / "patients_summary.csv"
GENDER_CATEGORIES_PATH = OUTPUT_DIR / "gender_categories.csv"
VISITS_SUMMARY_PATH = OUTPUT_DIR / "visits_summary.csv"
DECISION_LOG_PATH = OUTPUT_DIR / "decision_log.csv"
A1C_SUMMARY_PATH = OUTPUT_DIR / "a1c_summary.csv"
MERGE_AUDIT_PATH = OUTPUT_DIR / "merge_audit.csv"
SBP_WIDE_SAMPLE_PATH = OUTPUT_DIR / "sbp_wide_sample.csv"
ANALYSIS_SAMPLE_PATH = OUTPUT_DIR / "analysis_sample.csv"

# The patients whose rows the two extracts hold (README Tasks 4.3 and 5.2).
SAMPLE_IDS = ["P00001", "P00004", "P00088", "P00260", "P00446", "P00487", "P00631", "P00851",
              "P02354", "P03836", "P05763", "P07510", "P14287", "P16486", "P17390", "P30001"]

# The data is generated, not committed: generate_data.py writes the same tables on every computer.
manifest = json.loads((BASE_DIR / "data_manifest.json").read_text(encoding="utf-8"))
if not all((DATA_DIR / name).is_file() for name in manifest["files"]):
    print(subprocess.run([sys.executable, "generate_data.py"], cwd=BASE_DIR, check=True,
                         capture_output=True, text=True).stdout.strip())


def read_raw(path):
    """A Parquet file with the dtypes it stores; a CSV file with every column as text, blanks kept as written."""
    if path.suffix == ".parquet":
        return pd.read_parquet(path)
    return pd.read_csv(path, dtype="string", keep_default_na=False)


raw = {Path(name).stem: read_raw(DATA_DIR / name) for name in manifest["files"]}
# The checksum covers each table's contents, written out as CSV text.
for name, expected in manifest["files"].items():
    text = raw[Path(name).stem].to_csv(index=False, lineterminator="\n")
    assert hashlib.sha256(text.encode("utf-8")).hexdigest() == expected["sha256"], (
        f"data/{name} differs from the generated file; delete the data folder and run this cell again.")
print(f"Verified {len(manifest['files'])} data files")
raw_snapshot = {name: table.copy(deep=True) for name, table in raw.items()}
list(raw)

Python 3.13.15 | NumPy 2.3.3 | pandas 3.0.5
Verified 5 data files


['patients', 'sites', 'visits_north', 'visits_south', 'a1c_wide']

## Cumulative midterm checkpoint (human review)

**TODO:** Replace this text with an evidence map: one bullet for each of Lectures 01 to 06. For 01, 02, 04, 05, and 06, name one concrete technique or file and say in one sentence where this notebook or its outputs use it. For 03, complete the NumPy calculation after Task 4.3 and explain your axis choice in one sentence here. The lectures cover the command line and Python basics (01); Git, functions, and files (02); environments, shell pipelines, and NumPy (03); Jupyter and labeled pandas data (04); data contracts, cleaning, and validation (05); and joins, reshaping, and concatenation (06). Do not paste earlier solutions.
4. In 3.2 we are using descriptive statistics with regards to dates to get a min and a maximum
5. Section 2.3 we are using data validation to confirm that dates are correct and match a specific pattern. We are also using rules for systolic blood pressure.
6. Section 2.2. We are using vertical Concantenation to join the two data sets together.

## Task 1: Inventory the files

### 1.1 Count lines from the terminal

Make `output/file_inventory.txt` in the terminal (README Task 1.1), then paste the command you ran into the `%%bash` cell below.

In [2]:
%%bash
wc -l data/*.csv > output/file_inventory.txt

### 1.2 Profile each table with a function

In [3]:
# TODO: write a function, such as profile(name, table), that returns the four values in README
#Write a function that takes a data file's name and its table and returns a dictionary with four keys: table (the file name, such as patients.parquet), rows (data rows, not the header), columns, and exact_duplicate_rows (rows identical to an earlier row: table.duplicated().sum()). Call it once per file: loop over manifest["files"], a dictionary keyed by file name, so for name in manifest["files"] yields each name; pass each name with its table, raw[Path(name).stem]. Collect the dictionaries in a list, turn it into a DataFrame with pd.DataFrame(results) (one row per dictionary), and save it with to_csv(path, index=False).

#Self-check: each CSV file's rows is one less than its wc -l count in Task 1.1.
def profile(name, table):
    return {
        "table" : name,
        "rows": len(table),
        "columns": len(table.columns),
        "exact_duplicate_rows": table.duplicated().sum()
    }
# Task 1.2, call it once per name in manifest["files"] with raw[Path(name).stem], and save the results to PROFILE_PATH.
results = []
for name in manifest["files"]:
    table = raw[Path(name).stem]
    results.append(profile(name, table))
summary = pd.DataFrame(results)
summary.to_csv(PROFILE_PATH, index=False)

## Task 2: Contract, stack, and audit

### 2.1 Write the data contract (human review)

**TODO:** Replace this text with the contract: the intended grain and key (single or composite) of each cleaned input table, and the cardinality you expect for each Task 4.4 join. Describe visits after stacking and keeping the later recheck, sites after keeping current records, and the long-form A1c join keys. Note raw exact repeats, rechecks, and site versions that violate the intended cleaned keys (README Task 2.1).

Patients.parquet | one registered patient | key: patient id(single)|

sites.csv | One current record of the site(retired version were dropped) site record | key: site code(single)|

visits_north.csv | once clinic vist exported by the north office. This is done after exact repeats are dropped with the last repeat being kept | key: patient_id and visit(composite)|

visits_south.csv | once clinic vist exported by the south office. This is done after exact repeats are dropped with the last repeat being kept| key : patient_id and visit(composite)|

a1c_wide.parquet | one patient's central lab results| key: patient_id(single) |

#Still to do, name the cardinality.

### 2.2 Stack the two visit files

In [4]:
# TODO: rename wt_kg, add source_file to each visit table, and stack them, north first (README Task 2.2).
south = raw["visits_south"].copy(deep=True).rename(columns={"wt_kg": "weight_kg"})
north = raw["visits_north"].copy(deep=True)

north["source_file"] = "visits_north.csv"
south["source_file"] = "visits_south.csv"
visits_raw = pd.concat([north, south], ignore_index=True)
dataframe = pd.DataFrame(visits_raw)
print(dataframe)

      patient_id     visit  visit_date visit_time sbp_mmhg weight_kg  \
0         P00002  baseline  2026-02-20      09:50      132      85.2   
1         P00002    week12  2026-05-08      15:20      129      82.0   
2         P00003  baseline  2026-01-16      11:45      148      67.5   
3         P00003      wk12  2026-04-07      10:00      137      64.0   
4         P00004  baseline  2026-02-25      10:55      145     106.2   
...          ...       ...         ...        ...      ...       ...   
57366     P24976    week12  2026-05-14      10:50      153      78.3   
57367     P19276  baseline  2026-01-22      15:35      160     122.2   
57368     P21334    week12  2026-05-05      09:40      129      98.2   
57369     P03524  baseline  2026-02-06      10:05      160     117.6   
57370     P17358  baseline  2026-03-17      08:30      171     109.8   

            source_file  
0      visits_north.csv  
1      visits_north.csv  
2      visits_north.csv  
3      visits_north.csv  
4    

### 2.3 Audit the raw tables

In [5]:
# TODO: count the 10 issues in README Task 2.3, in order, as a DataFrame with the columns issue and
# count, and save it to AUDIT_PATH.
patients = raw["patients"].copy(deep=True)
sites = raw["sites"].copy(deep=True)
visits_raw_c = visits_raw.copy(deep=True)


#gender variables
gender = patients["gender"].astype("string")
gender_stripped = gender.str.strip().fillna("")
gender_invalid = (~gender.isin(["M", "F", "X"]))& (gender_stripped != "")
empty_gender = gender_stripped == ""

#Birth_sentinel
birth_sentinel = pd.to_numeric(patients["birth_year"], errors="coerce") == 9999

#SIte code normalization patients
site_code = patients["site_code"].astype("string").fillna("")
site_normalized = site_code.str.upper().str.replace(r"[ -]", "",regex=True)
site_invalid = site_code != site_normalized

#Sites where row whose site code also appears on another row.
site_duplicated = sites.duplicated(subset=["site_code"], keep=False)

#Rows whose visit as written is not baseline or week12
visit_invalid = ~visits_raw_c["visit"].isin(["baseline", "week12"])


#BP values
#rows where sbp is 999
sbp = pd.to_numeric(visits_raw_c["sbp_mmhg"].astype("string").str.strip(), errors="coerce")
sbp_sentinel = sbp == 999

#sbp out of range
sbp_outside_range = ((sbp<70) | (sbp>250)) & (sbp != 999)


#date parse failures
exact = visits_raw_c["visit_date"].str.fullmatch(r'[0-9]{4}-[0-9]{2}-[0-9]{2}')
date_parsed_f = pd.to_datetime(visits_raw_c["visit_date"].where(exact), format='%Y-%m-%d', errors='coerce')


#Rows in repeat patient_id and visit pairs
visit_duplicated = visits_raw_c.duplicated(subset = ["patient_id", "visit"], keep=False)
issue_audit = pd.DataFrame(
    [
        ("patients: gender values that are not a code", gender_invalid.sum()),
        ("patients: empty gender", empty_gender.sum()),
        ("patients: birth_year sentinel 9999", birth_sentinel.sum()),
        ("patients: site codes needing normalization", site_invalid.sum()),
        ("sites: rows with a repeated site_code", site_duplicated.sum()),
        ("visits: visit labels that are not baseline or week12", visit_invalid.sum()),
        ("visits: sbp sentinel 999", sbp_sentinel.sum()),
        ("visits: sbp outside 70 through 250", sbp_outside_range.sum()),
        ("visits: date parse failures", date_parsed_f.isna().sum()),
        ("visits: rows in repeated patient_id and visit pairs", visit_duplicated.sum())



    ],
    columns=["issue", "count"]
)
print(issue_audit)

assert all(raw[name].equals(raw_snapshot[name]) for name in raw), "Auditing must not change raw."

issue_audit.to_csv(AUDIT_PATH, index=False)


                                               issue  count
0        patients: gender values that are not a code   6968
1                             patients: empty gender    486
2                 patients: birth_year sentinel 9999    371
3         patients: site codes needing normalization   1811
4              sites: rows with a repeated site_code      4
5  visits: visit labels that are not baseline or ...   2309
6                           visits: sbp sentinel 999    341
7                 visits: sbp outside 70 through 250    116
8                        visits: date parse failures    169
9  visits: rows in repeated patient_id and visit ...    288


## Task 3: Clean

### 3.1 Clean the patients

In [6]:
# TODO: list the gender spellings with value_counts(), then build patients_clean from
# raw["patients"].copy(deep=True) with steps 1 to 5 in README Task 3.1.
patients_clean = raw["patients"].copy(deep=True)


#Drop duplicates
patients_clean = patients_clean.drop_duplicates(keep="first")

#Gender cleaning
print(patients_clean["gender"].value_counts())
gender_map = {
    "f" :"F",
    "female" : "F",
    "woman" : "F",
    "m" :"M",
    "male" : "M",
    "man" : "M",
    "x" : "X",
    "nonbinary" : "X",
    "non-binary" : "X",
    "nb" : "X"
}
gender_clean = (
    patients_clean["gender"].astype("string").str.strip().str.lower().map(gender_map)
)
patients_clean["gender"] = gender_clean.astype("category")

#Birth year cleaning
birth = pd.to_numeric(patients_clean["birth_year"], errors="coerce")
patients_clean["birth_year"] = birth.mask(birth == 9999).astype("Int64")

#SIte code uppercase and strip
patients_clean["site_code"] = (patients_clean["site_code"].astype("string").str.upper().str.replace("-", "").str.replace(" ", ""))

#Resetting the index:
patients_clean = patients_clean.reset_index(drop=True)

print(patients_clean["gender"].cat.categories)


gender
F             11230
M             10952
male           1038
female         1018
f               622
Male            621
Female          577
m               537
woman           399
                381
X               372
man             329
unknown         282
MALE            255
FEMALE          231
 M              223
 F              205
Woman           131
nonbinary       118
                104
NB              103
Unknown          96
non-binary       80
x                62
Non-Binary       34
Name: count, dtype: int64[pyarrow]
Index(['F', 'M', 'X'], dtype='str')


In [7]:
# TODO: show patients_clean["gender"].cat.categories, save the patients summary to
# PATIENTS_SUMMARY_PATH, and save the gender value_counts(dropna=False) with a code column from
# .cat.codes to GENDER_CATEGORIES_PATH, as in README Task 3.1.
print(patients_clean["gender"].cat.categories)
patients_summary = pd.DataFrame({
    "metric": ["patients", "birth_year missing", "birth_year min", "birth_year max", "site_code values", "patients at N01", "patients at S02", "patients at E04" ],
    "value": [len(patients_clean), patients_clean["birth_year"].isna().sum(), patients_clean["birth_year"].min(), patients_clean["birth_year"].max(), patients_clean["site_code"].nunique(), (patients_clean["site_code"] == "N01").sum(), (patients_clean["site_code"] == "S02").sum(), (patients_clean["site_code"] == "E04").sum()]
})
patients_summary.to_csv(PATIENTS_SUMMARY_PATH, index=False)


gender_counts = patients_clean["gender"].value_counts(dropna=False).reset_index()
gender_counts["code"] = gender_counts["gender"].cat.codes
gender_counts.to_csv(GENDER_CATEGORIES_PATH, index=False)


Index(['F', 'M', 'X'], dtype='str')


### 3.2 Clean the visits

In [8]:
# TODO: build visits_clean from visits_raw.copy(deep=True) with steps 1 to 5 in README Task 3.2.
visits_clean = visits_raw.copy(deep=True)

#Drop duplicates keeping first
visits_clean = visits_clean.drop_duplicates(keep="first")

#List the spelling then strip and lowercase them and map to baseline or week12.

print(visits_clean["visit"].value_counts())
visits_map = {
    "baseline" : "baseline",
    "bl" : "baseline",
    "week12" : "week12",
    "week 12" : "week12",
    "wk12" : "week12",
    " week12" :"week12"
}

visit_cleaned = (
    visits_clean["visit"].astype("string").str.strip().str.lower().map(visits_map)
)
visits_clean["visit"] = visit_cleaned

print(visits_clean["visit"].value_counts())

#BP Conversion:
sbp = pd.to_numeric(visits_clean["sbp_mmhg"], errors="coerce")
visits_clean["sbp_mmhg"] = sbp.where(sbp.between(70, 250)).astype("Int64")

#Weight conversion, we leave behind missing values.
visits_clean["weight_kg"] = pd.to_numeric(visits_clean["weight_kg"], errors="coerce")

#Visit date cleaning
date_text = visits_clean["visit_date"].astype("string")
exact = date_text.str.fullmatch(r'[0-9]{4}-[0-9]{2}-[0-9]{2}').fillna(False).astype(bool)
visits_clean["visit_date"] = pd.to_datetime(date_text.where(exact), format='%Y-%m-%d', errors='coerce')




visit
baseline     28496
week12       26498
BASELINE       255
Baseline       242
bl             235
baseline       228
BL             220
wk12           204
 week12        198
Week 12        194
WEEK12         186
Wk12           177
week 12        168
Name: count, dtype: int64[pyarrow]
visit
baseline    29676
week12      27625
Name: count, dtype: int64


In [9]:
# TODO: step 6, show the repeated patient_id and visit pairs, then keep the later reading.

#Check where patient_id AND visit pair is repeated

#Rows in repeat patient_id and visit pairs
dup_mask = visits_clean.duplicated(subset = ["patient_id", "visit"], keep=False)
print(visits_clean[dup_mask].sort_values(["patient_id", "visit", "visit_time"]))
#now actually dropping them:
visits_clean = (visits_clean.sort_values("visit_time").drop_duplicates(subset=["patient_id", "visit"], keep = "last"))


      patient_id     visit visit_date visit_time  sbp_mmhg  weight_kg  \
26298     P00001  baseline 2026-03-06      14:15       141       52.4   
26297     P00001  baseline 2026-03-06      14:40       146       52.4   
4         P00004  baseline 2026-02-25      10:55       145      106.2   
5         P00004  baseline 2026-02-25      11:15       142      106.2   
84        P00088  baseline 2026-03-10      09:30       153       68.2   
...          ...       ...        ...        ...       ...        ...   
55473     P28232  baseline 2026-01-12      10:10       159       61.0   
56249     P28953  baseline 2026-01-21      10:30       153       74.8   
56248     P28953  baseline 2026-01-21      10:45       151       74.8   
25464     P29170  baseline 2026-03-02      11:20       130       78.0   
25465     P29170  baseline 2026-03-02      11:45       128       78.0   

            source_file  
26298  visits_south.csv  
26297  visits_south.csv  
4      visits_north.csv  
5      visits_north

In [21]:
# TODO: step 7, sort, renumber, keep the six columns, then save the visits summary to
# VISITS_SUMMARY_PATH (README Task 3.2).

#Sort by patient_id and visit
visits_clean = (visits_clean
                .sort_values(["patient_id", "visit"])
                .reset_index(drop=True)
                [["patient_id", "visit", "visit_date", "sbp_mmhg", "weight_kg", "source_file"]]
)

visits_summary = pd.DataFrame([
   ("visits", len(visits_clean)),
   ("baseline visits", (visits_clean["visit"] == "baseline").sum()),
   ("week12 visits", (visits_clean["visit"] == "week12").sum()),
   ("visit_date missing", visits_clean["visit_date"].isna().sum()),
   ("visit_date min", visits_clean["visit_date"].min()),
   ("visit_date max", visits_clean["visit_date"].max()),
   ("sbp_mmhg missing", visits_clean["sbp_mmhg"].isna().sum()),
   ("sbp_mmhg min", visits_clean["sbp_mmhg"].min()),
   ("sbp_mmhg max", visits_clean["sbp_mmhg"].max()),
   ("weight_kg missing", visits_clean["weight_kg"].isna().sum()),
   ("weight_kg mean", round(float(visits_clean["weight_kg"].mean()), 2)),
   ("visits from visits_north.csv", int((visits_clean["source_file"] == "visits_north.csv").sum())),
   ("visits from visits_south.csv", int((visits_clean["source_file"] == "visits_south.csv").sum())),
],
columns=["metric", "value"]
)

visits_summary.to_csv(VISITS_SUMMARY_PATH, index=False)

### 3.3 Record the decisions

In [ ]:
# TODO: one row per issue in README Task 3.3, with table and issue copied, and your own
# decision and one-sentence reason, then save it to DECISION_LOG_PATH.
decision_table = pd.DataFrame([
    ("patients", "exact duplicate rows", 
     "removed the duplicated rows specifically removing the first",
     "This was done because there were vists were "
     "BP were repeated on the same date shortly after ther initial and keeping the last is more accurate"),
    ("patients", "gender spellings that are not a code", 
     "Used mapping to ensure that all of the differnent ways of input male, "
     "female and nonbinary were convereted into usable data",
      "This was done so that analysis can be done on the information as well as filtering." ),
    ("patients", "site codes needing normalization",
     "site codes were converted to lower case, had spaces and hypen removed",
     "This was done in order to allow us to properly clairfy which sites we had patients"
     "seen at given data entry was performed in multiple ways"),
     (
         "patients", "birth_year sentinel 9999", 
         "Removed birth year 9999 from the cleaned data through masking",
         "This was done since 9999 is not a valid date and likely represents data that was N/A or not collected"
     ),
     (
         "visits", "weight column named wt_kg in visits_south.csv",
         "we renamed weight column to wt_kg",
         "We did this so we can stack the data from visits north and visits south, if we had not done this they would not have the same columns present" 
     ),
     (
         "visits", "sbp sentinel 999 and reading outside 70 thorugh 250",
         "We converted to numeric data and selected data that only was in between 70-250 to keep with .where",
         "We did this to remove NA data which was coded as 999, values that were unliekly real(eg a BP of <70 you would see in a"
         "critically ill patient, and BP of 250 would be a patient that was actively having a stroke from such high bp)"
         
     ),
     (
         "visits", "impossible and loosely written visit dates",
         "we filtered dates with fullmatch and pd.to_datetime with confirming it was is the format y, m, d",
         "We did this to ensure that we have realstic dates showing up"
     ),


    ],
    columns=["table", "issue", "decision", "reason"])
decision_table

,table,issue,decision,reason
0,patients,exact duplicate rows,removed the duplicated rows specifically remov...,This was done because there were vists were BP...
1,patients,gender spellings that are not a code,Used mapping to ensure that all of the differn...,This was done so that analysis can be done on ...
2,patients,site codes needing normalization,"site codes were converted to lower case, had s...",This was done in order to allow us to properly...


## Task 4: Reshape and join

### 4.1 Melt the A1c results to long form

In [12]:
# TODO: melt raw["a1c_wide"], map the visit labels, drop the rows with no result,
# then save the per-visit summary to A1C_SUMMARY_PATH (README Task 4.1).
a1c_long = None

### 4.2 Check the keys before joining

In [13]:
# TODO: steps 1 and 2, catch the MergeError, show the repeated site rows, and build sites_current.
sites = raw["sites"]
sites_current = None

In [14]:
# TODO: step 3, outer merge visits_clean and a1c_long with indicator=True and show the _merge counts,
# then save the seven counts in README Task 4.2 to MERGE_AUDIT_PATH.

### 4.3 Pivot blood pressure to one row per patient

In [15]:
# TODO: pivot visits_clean, reset the index, clear columns.name, add sbp_change, then save the
# SAMPLE_IDS rows to SBP_WIDE_SAMPLE_PATH (README Task 4.3).
sbp_wide = None

#### NumPy checkpoint (Lecture 03, human review)

From the full `sbp_wide`, select `baseline` and `week12` in that order, keeping only patients with both readings. Name this two-column table `complete_pairs`; `np.array(complete_pairs, dtype=float)` converts it to a NumPy array. Display its shape and calculate one NumPy mean per visit across patients. Save the outputs and explain your axis choice in the cumulative checkpoint's Lecture 03 bullet. No extra output file.

In [16]:
# TODO: select complete pairs, convert to a NumPy array, and display its shape and two per-visit means.

### 4.4 Join the analysis table

In [17]:
# TODO: three left merges onto visits_clean, each with validate=, then the columns and dtypes
# in README Task 4.4. A MergeError means an earlier step left a problem: see HINTS.md.
analysis = None

## Task 5: Validate, save, and read back

### 5.1 Validate before saving

In [18]:
# TODO: collect named True/False checks in a Series (README Task 5.1), display it, and stop with
# assert checks.all(), checks[~checks] before the analysis extract is saved.
# A failed check means an earlier step left a problem: print checks[~checks] and see HINTS.md.

### 5.2 Save the analysis extract

In [19]:
# TODO: keep the analysis rows whose patient_id is in SAMPLE_IDS, renumber them, and save them to
# ANALYSIS_SAMPLE_PATH with index=False (README Task 5.2).

### 5.3 Read the file back

In [20]:
# TODO: read ANALYSIS_SAMPLE_PATH back with its dtypes and parse_dates=["visit_date"], and print
# whether it equals analysis_sample.